<a href="https://colab.research.google.com/github/AjayJangid09/AI_Tutor/blob/main/updated_ai_tutor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install torch transformers sentence-transformers
!pip install langchain langchain-community langchain-text-splitters langchain-huggingface
!pip install chromadb rank_bm25
!pip install presidio-analyzer presidio-anonymizer
!pip install pymupdf pypdf2 python-docx
!pip install rouge-score nltk scikit-learn
!pip install spacy && python -m spacy download en_core_web_sm
!pip install tqdm colorama rich

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.3/596.3 kB 14.5 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-huggingface 1.2.0 requires huggingface-hub<1.0.0,>=0.33.4, but you have huggingface-hub 1.5.0 which is incompatible.
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.5.0
    Uninstalling huggingface_hub-1.5.0:
      Successfully uninstalled huggingface_hub-1.5.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installe

In [2]:
!pip install -U transformers huggingface_hub sentence-transformers langchain-huggingface

  Using cached huggingface_hub-1.5.0-py3-none-any.whl.metadata (13 kB)
INFO: pip is looking at multiple versions of langchain-huggingface to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain_huggingface-1.2.0-py3-none-any.whl.metadata (2.8 kB)
INFO: pip is still looking at multiple versions of langchain-huggingface to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 62.3 MB/s eta 0:00:00
Using cached huggingface_hub-1.5.0-py3-none-any.whl (596 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 458.9/458.9 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.4 MB/s eta 0:00:00
  Attempting uninstall: packaging
    Found existing installation: packaging 26.0
    Uninstalling packaging-26.0:
      Successfully uninstalled packaging-26.0
  Attempting uninstall: langchain-core
    Found existing installatio

In [1]:
!pip install -U langchain langchain-core langchain-community

  Using cached langchain_core-1.2.16-py3-none-any.whl.metadata (4.4 kB)
Using cached langchain_core-1.2.16-py3-none-any.whl (502 kB)
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.83
    Uninstalling langchain-core-0.3.83:
      Successfully uninstalled langchain-core-0.3.83
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-huggingface 0.3.1 requires langchain-core<1.0.0,>=0.3.70, but you have langchain-core 1.2.16 which is incompatible.


In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║         PRODUCTION-GRADE AI TUTOR SYSTEM — Full RAG Pipeline               ║
║         Open-Source | PDF Ingestion | Hybrid Search | Guardrails            ║
║         Prompt Injection & Toxicity Guardrails | Eval Suite | PII Scrub     ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""

import os
import re
import json
import time
import warnings
import hashlib
import logging
import threading # <-- Naya import

from pathlib import Path
from typing import List, Dict, Tuple, Optional, Any
from dataclasses import dataclass, field
from datetime import datetime

import torch
import numpy as np

# ─── Silence Noisy Libraries & Background Threads ──────────────────────────────
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import transformers
logging.getLogger("presidio-analyzer").setLevel(logging.ERROR)
transformers.logging.set_verbosity_error()

# Naya block: Hugging Face ke crash hone wale background thread ko mute karne ke liye
_original_excepthook = threading.excepthook
def custom_excepthook(args):
    if args.thread is not None and args.thread.name == "Thread-auto_conversion":
        pass # Chup-chaap ignore kar do
    else:
        _original_excepthook(args) # Baki koi real error ho toh print karo
threading.excepthook = custom_excepthook
# ─────────────────────────────────────────────────────────────────────────────

# ─── Langchain ────────────────────────────────────────────────────────────────
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever

# ─── Transformers ─────────────────────────────────────────────────────────────
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    pipeline,
    StoppingCriteria,
    StoppingCriteriaList,
)
from sentence_transformers import CrossEncoder

# ─── PII / Guardrails ─────────────────────────────────────────────────────────
from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine
from presidio_analyzer.nlp_engine import NlpEngineProvider

# ─── PDF / Document Loaders ───────────────────────────────────────────────────
try:
    import fitz  # PyMuPDF — best quality PDF extraction
    PDF_BACKEND = "pymupdf"
except ImportError:
    try:
        from PyPDF2 import PdfReader
        PDF_BACKEND = "pypdf2"
    except ImportError:
        PDF_BACKEND = "none"

# ─── Evaluation ───────────────────────────────────────────────────────────────
from rouge_score import rouge_scorer
import nltk
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)
try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt_tab', quiet=True)

# ─── Logging setup ────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler("tutor_system.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger("TutorSystem")


# ══════════════════════════════════════════════════════════════════════════════
# DATA CLASSES
# ══════════════════════════════════════════════════════════════════════════════

@dataclass
class TutorConfig:
    """Central configuration for the tutor system."""
    # Models
    embedding_model: str = "sentence-transformers/all-MiniLM-L6-v2"
    llm_model: str = "Qwen/Qwen2.5-0.5B-Instruct"
    reranker_model: str = "cross-encoder/ms-marco-MiniLM-L-6-v2"

    # Guardrail Models
    nli_model: str = "cross-encoder/nli-distilroberta-base"
    toxicity_model: str = "s-nlp/roberta_toxicity_classifier"
    injection_model: str = "deepset/deberta-v3-base-injection"

    # Chunking
    chunk_size: int = 512
    chunk_overlap: int = 64

    # Retrieval
    dense_top_k: int = 4
    sparse_top_k: int = 4
    reranker_top_k: int = 3

    # Generation
    max_new_tokens: int = 300
    temperature: float = 0.4
    top_p: float = 0.9
    repetition_penalty: float = 1.15

    # Guardrails Config
    nli_threshold: float = 0.5      # Entailment probability threshold
    min_answer_length: int = 10     # Reject trivially short answers

    # Storage
    chroma_dir: str = "./chroma_db"
    chroma_collection: str = "tutor_kb"

    # Device
    device: str = "cuda" if torch.cuda.is_available() else "cpu"


@dataclass
class QueryResult:
    """Structured result from a tutor query."""
    query: str
    sanitized_query: str
    pii_detected: bool
    retrieved_docs: List[Document]
    context_used: str
    raw_answer: str
    final_answer: str
    guardrail_passed: bool
    guardrail_reason: str
    nli_score: float
    retrieval_scores: List[float]
    latency_ms: float
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())


@dataclass
class EvalResult:
    """Evaluation metrics for a single QA pair."""
    question: str
    reference_answer: str
    predicted_answer: str
    rouge1: float
    rouge2: float
    rougeL: float
    exact_match: float
    faithfulness_score: float   # NLI-based
    answer_relevance: float     # Cosine sim between question & answer embeddings
    context_recall: float       # Did retrieved docs contain the answer?
    passed_guardrail: bool


# ══════════════════════════════════════════════════════════════════════════════
# MODULE 1: PDF / DOCUMENT LOADER
# ══════════════════════════════════════════════════════════════════════════════

class DocumentLoader:
    """
    Loads PDFs, plain text, and markdown files into LangChain Documents.
    Uses PyMuPDF for best PDF quality, falls back to PyPDF2.
    """
    def __init__(self):
        logger.info(f"   📄 PDF backend: {PDF_BACKEND}")

    def load_pdf_pymupdf(self, path: str) -> List[Document]:
        docs = []
        with fitz.open(path) as pdf:
            for i, page in enumerate(pdf):
                text = page.get_text("text").strip()
                if text:
                    docs.append(Document(
                        page_content=text,
                        metadata={"source": path, "page": i + 1, "type": "pdf"}
                    ))
        logger.info(f"   Loaded {len(docs)} pages from {Path(path).name}")
        return docs

    def load_pdf_pypdf2(self, path: str) -> List[Document]:
        docs = []
        reader = PdfReader(path)
        for i, page in enumerate(reader.pages):
            text = page.extract_text()
            if text and text.strip():
                docs.append(Document(
                    page_content=text.strip(),
                    metadata={"source": path, "page": i + 1, "type": "pdf"}
                ))
        return docs

    def load_text(self, path: str) -> List[Document]:
        content = Path(path).read_text(encoding="utf-8", errors="ignore")
        return [Document(page_content=content, metadata={"source": path, "type": "text"})]

    def load(self, path: str) -> List[Document]:
        p = Path(path)
        if not p.exists():
            raise FileNotFoundError(f"File not found: {path}")
        ext = p.suffix.lower()
        if ext == ".pdf":
            if PDF_BACKEND == "pymupdf":
                return self.load_pdf_pymupdf(path)
            elif PDF_BACKEND == "pypdf2":
                return self.load_pdf_pypdf2(path)
            else:
                raise RuntimeError("No PDF library found. Install pymupdf or pypdf2.")
        elif ext in [".txt", ".md", ".rst"]:
            return self.load_text(path)
        else:
            raise ValueError(f"Unsupported file type: {ext}")

    def load_multiple(self, paths: List[str]) -> List[Document]:
        all_docs = []
        for path in paths:
            all_docs.extend(self.load(path))
        return all_docs


# ══════════════════════════════════════════════════════════════════════════════
# MODULE 2: GUARDRAILS (PII + Injection + Toxicity + Hallucination + Length)
# ══════════════════════════════════════════════════════════════════════════════

class GuardrailSystem:
    """
    Multi-layer guardrail system:
      1. PII detection + anonymization (Presidio)
      2. Prompt Injection detection (Input)
      3. Toxicity detection (Input & Output)
      4. Length-based trivial answer rejection
      5. NLI-based hallucination detection (CrossEncoder)
    """

    PII_ENTITIES = [
        "PHONE_NUMBER", "EMAIL_ADDRESS", "CREDIT_CARD",
        "IBAN_CODE", "IP_ADDRESS","NRP"]

    def __init__(self, config: TutorConfig):
        logger.info("   🛡️  Loading Guardrails...")
        self.config = config

        # PII setup
        cfg = {
            "nlp_engine_name": "spacy",
            "models": [{"lang_code": "en", "model_name": "en_core_web_sm"}]
        }
        provider = NlpEngineProvider(nlp_configuration=cfg)
        self.analyzer = AnalyzerEngine(
            nlp_engine=provider.create_engine(),
            supported_languages=["en"]
        )
        self.anonymizer = AnonymizerEngine()

        # Hallucination setup
        self.nli = CrossEncoder(config.nli_model, device=config.device)

        # Toxicity and Prompt Injection Classifiers
        self.toxicity_pipe = pipeline("text-classification", model=config.toxicity_model, device=config.device)
        self.injection_pipe = pipeline("text-classification", model=config.injection_model, device=config.device)

        logger.info("   ✅ Guardrails loaded (PII, Toxicity, Injection, NLI).")

    def sanitize_pii(self, query: str) -> Tuple[str, bool, List]:
        """Detect and anonymize PII in user input."""
        results = self.analyzer.analyze(
            query, entities=self.PII_ENTITIES, language='en'
        )
        if results:
            anonymized = self.anonymizer.anonymize(query, analyzer_results=results)
            return anonymized.text, True, results
        return query, False, []

    def check_input_safety(self, query: str) -> Tuple[bool, str]:
        """Check for prompt injection and toxic input."""
        # 1. Prompt Injection Check
        inj_res = self.injection_pipe(query, truncation=True, max_length=512)
        if inj_res[0]['label'] == 'INJECTION' and inj_res[0]['score'] > 0.6:
            return False, f"Prompt injection detected (score: {inj_res[0]['score']:.2f})"

        # 2. Toxicity Check
        tox_res = self.toxicity_pipe(query, truncation=True, max_length=512)
        if tox_res[0]['label'] == 'toxic' and tox_res[0]['score'] > 0.6:
            return False, f"Toxic input detected (score: {tox_res[0]['score']:.2f})"

        return True, "Safe"

    def check_answer_length(self, answer: str) -> Tuple[bool, str]:
        """Reject trivially short answers."""
        stripped = answer.strip()
        if len(stripped) < self.config.min_answer_length:
            return False, f"Answer too short ({len(stripped)} chars)"
        if stripped.lower() in ["i don't know", "i don't know.", "unknown", "n/a", ""]:
            return False, "Non-informative answer"
        return True, "ok"

    def check_hallucination(self, context: str, answer: str) -> Tuple[bool, float, str]:
            """
            NLI-based faithfulness check.
            Labels: 0=Contradiction, 1=Neutral, 2=Entailment
            """
            if not context.strip():
                return True, 0.5, "no_context"

            # FIX 1: Clean Metadata. NLI models get confused by [Source 1: ...] tags.
            # Ye regex saare source tags ko hata dega taaki model sirf plain text padhe.
            clean_context = re.sub(r'\[Source \d+:.*?\]\n', '', context)

            # FIX 2: Increase Truncation Limit.
            # Cross-Encoders can usually handle up to 512 tokens (~2500-3000 chars).
            ctx_truncated = clean_context[:3000]
            ans_truncated = answer[:500]

            scores = self.nli.predict([(ctx_truncated, ans_truncated)])
            probs = torch.softmax(torch.tensor(scores[0]), dim=0).numpy()

            contradiction_prob = float(probs[0])
            entailment_prob = float(probs[2])
            nli_score = entailment_prob

            # FIX 3: Relax Threshold slightly. NLI models can be overly strict.
            # Changed from 0.7 to 0.85 to avoid false positives on normal answers.
            if contradiction_prob > 0.85:
                return False, nli_score, f"contradiction (p={contradiction_prob:.2f})"

            return True, nli_score, f"entailment={entailment_prob:.2f}"

    def validate_answer(self, context: str, answer: str) -> Tuple[bool, str, float]:
        """Run all answer validation checks (Length, Toxicity, Hallucination)."""
        # 1. Length check
        ok, reason = self.check_answer_length(answer)
        if not ok:
            return False, reason, 0.0

        # 2. Toxicity Check (Output Guardrail)
        tox_res = self.toxicity_pipe(answer, truncation=True, max_length=512)
        if tox_res[0]['label'] == 'toxic' and tox_res[0]['score'] > 0.6:
            return False, f"Toxic output generated (score: {tox_res[0]['score']:.2f})", 0.0

        # 3. NLI hallucination check
        faithful, score, detail = self.check_hallucination(context, answer)
        if not faithful:
            return False, f"Hallucination detected: {detail}", score

        return True, "all_checks_passed", score


# ══════════════════════════════════════════════════════════════════════════════
# MODULE 3: KNOWLEDGE BASE (Hybrid Dense + Sparse + Reranker)
# ══════════════════════════════════════════════════════════════════════════════

class HybridEnsembleRetriever:
    """Combines dense and sparse retrievers with Reciprocal Rank Fusion & cross-encoder reranking."""

    def __init__(self, dense_retriever, sparse_retriever, reranker: CrossEncoder, config: TutorConfig):
        self.dense = dense_retriever
        self.sparse = sparse_retriever
        self.reranker = reranker
        self.config = config

    def _rrf_fusion(self, dense_docs: List[Document], sparse_docs: List[Document], k: int = 60) -> List[Document]:
        scores: Dict[str, float] = {}
        doc_map: Dict[str, Document] = {}

        for rank, doc in enumerate(dense_docs):
            key = hashlib.md5(doc.page_content.encode()).hexdigest()
            scores[key] = scores.get(key, 0) + 1.0 / (k + rank + 1)
            doc_map[key] = doc

        for rank, doc in enumerate(sparse_docs):
            key = hashlib.md5(doc.page_content.encode()).hexdigest()
            scores[key] = scores.get(key, 0) + 1.0 / (k + rank + 1)
            doc_map[key] = doc

        sorted_keys = sorted(scores, key=lambda x: scores[x], reverse=True)
        return [doc_map[k] for k in sorted_keys]

    def invoke(self, query: str) -> Tuple[List[Document], List[float]]:
        dense_docs = self.dense.invoke(query)
        sparse_docs = self.sparse.invoke(query)
        merged = self._rrf_fusion(dense_docs, sparse_docs)

        if not merged:
            return [], []

        pairs = [(query, doc.page_content[:512]) for doc in merged]
        rerank_scores = self.reranker.predict(pairs)

        ranked = sorted(zip(merged, rerank_scores), key=lambda x: x[1], reverse=True)
        top_docs = [d for d, _ in ranked[:self.config.reranker_top_k]]
        top_scores = [float(s) for _, s in ranked[:self.config.reranker_top_k]]

        return top_docs, top_scores


class KnowledgeBase:
    def __init__(self, config: TutorConfig):
        logger.info("   📚 Loading Knowledge Base...")
        self.config = config

        self.embeddings = HuggingFaceEmbeddings(
            model_name=config.embedding_model,
            model_kwargs={"device": config.device},
            encode_kwargs={"normalize_embeddings": True}
        )

        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=config.chunk_size,
            chunk_overlap=config.chunk_overlap,
            separators=["\n\n", "\n", ". ", "! ", "? ", " ", ""],
            keep_separator=True,
        )

        self.reranker = CrossEncoder(config.reranker_model, device=config.device)
        self.retriever: Optional[HybridEnsembleRetriever] = None
        self._chunks: List[Document] = []
        logger.info("   ✅ Knowledge Base ready.")

    def ingest(self, docs: List[Document], persist: bool = True) -> int:
        logger.info(f"   📥 Ingesting {len(docs)} documents...")

        cleaned = [Document(
            page_content=self._clean_text(d.page_content),
            metadata=d.metadata
        ) for d in docs if len(d.page_content.strip()) > 50]

        chunks = self.splitter.split_documents(cleaned)
        self._chunks = chunks
        logger.info(f"   ✂️  Created {len(chunks)} chunks.")

        if not chunks:
            raise ValueError("No usable chunks after splitting. Check your documents.")

        chroma_dir = self.config.chroma_dir if persist else None
        db = Chroma.from_documents(
            chunks,
            self.embeddings,
            collection_name=self.config.chroma_collection,
            persist_directory=chroma_dir
        )
        dense_ret = db.as_retriever(
            search_type="mmr",
            search_kwargs={"k": self.config.dense_top_k, "fetch_k": 10}
        )

        sparse_ret = BM25Retriever.from_documents(chunks)
        sparse_ret.k = self.config.sparse_top_k

        self.retriever = HybridEnsembleRetriever(dense_ret, sparse_ret, self.reranker, self.config)
        logger.info(f"   ✅ Indexed {len(chunks)} chunks (Dense + BM25 + Reranker).")
        return len(chunks)

    @staticmethod
    def _clean_text(text: str) -> str:
        text = re.sub(r'\s+', ' ', text)
        text = re.sub(r'\n{3,}', '\n\n', text)
        text = re.sub(r'[^\x00-\x7F]+', ' ', text)
        return text.strip()


# ══════════════════════════════════════════════════════════════════════════════
# MODULE 4: LLM ENGINE (Qwen2.5 with Structured Prompting)
# ══════════════════════════════════════════════════════════════════════════════

class StopOnTokens(StoppingCriteria):
    def __init__(self, stop_ids: List[int]):
        self.stop_ids = stop_ids

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor, **kwargs) -> bool:
        for stop_id in self.stop_ids:
            if input_ids[0][-1] == stop_id:
                return True
        return False


class LLMEngine:
    SYSTEM_PROMPT = """You are an expert academic tutor. Your job is to answer student questions
accurately and clearly using ONLY the provided context.

Rules:
1. Base your answer EXCLUSIVELY on the given context.
2. If the context does not contain enough information, say: "The provided material does not cover this topic."
3. Do NOT make up facts or use outside knowledge.
4. Be concise but complete. Aim for 2-4 sentences.
5. If quoting directly, use quotation marks.
6. Structure: Brief answer → Supporting detail → Example (if available).
"""

    def __init__(self, config: TutorConfig):
        logger.info(f"   🤖 Loading LLM: {config.llm_model}...")
        self.config = config

        self.tokenizer = AutoTokenizer.from_pretrained(config.llm_model, trust_remote_code=True)
        self.model = AutoModelForCausalLM.from_pretrained(
            config.llm_model,
            torch_dtype=torch.float16 if config.device != "cpu" else torch.float32,
            device_map="auto",
            trust_remote_code=True
        )
        self.model.eval()

        stop_token_ids = []
        for token in ["<|im_end|>", "<|endoftext|>", "</s>"]:
            ids = self.tokenizer.encode(token, add_special_tokens=False)
            if ids: stop_token_ids.extend(ids)

        self.pipe = pipeline(
            "text-generation",
            model=self.model,
            tokenizer=self.tokenizer,
            max_new_tokens=config.max_new_tokens,
            max_length=None, # <-- ADDED TO PREVENT WARNING
            temperature=config.temperature,
            top_p=config.top_p,
            repetition_penalty=config.repetition_penalty,
            do_sample=True,
            stopping_criteria=StoppingCriteriaList([StopOnTokens(stop_token_ids)])
        )
        logger.info(f"   ✅ LLM loaded on {config.device}.")

    def build_prompt(self, context: str, query: str, history: List[Dict] = None) -> str:
        messages = [{"role": "system", "content": self.SYSTEM_PROMPT}]
        if history:
            for turn in history[-4:]:
                messages.append(turn)

        messages.append({
            "role": "user",
            "content": f"CONTEXT:\n{context}\n\nQUESTION: {query}"
        })
        return self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    def generate(self, context: str, query: str, history: List[Dict] = None) -> str:
        prompt = self.build_prompt(context, query, history)
        with torch.no_grad():
            output = self.pipe(prompt)

        full_text = output[0]["generated_text"]
        for marker in ["<|im_start|>assistant\n", "assistant\n", "ANSWER:"]:
            if marker in full_text:
                full_text = full_text.split(marker)[-1]
                break

        for stop in ["<|im_end|>", "<|endoftext|>", "</s>"]:
            full_text = full_text.replace(stop, "")

        return full_text.strip()


# ══════════════════════════════════════════════════════════════════════════════
# MODULE 5: EVALUATION SUITE
# ══════════════════════════════════════════════════════════════════════════════

class EvaluationSuite:
    def __init__(self, config: TutorConfig, embeddings: HuggingFaceEmbeddings):
        self.rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
        self.nli = CrossEncoder(config.nli_model, device=config.device)
        self.embeddings = embeddings
        self.config = config

    def _rouge_scores(self, reference: str, prediction: str) -> Dict[str, float]:
        scores = self.rouge.score(reference.lower(), prediction.lower())
        return {
            "rouge1": scores["rouge1"].fmeasure,
            "rouge2": scores["rouge2"].fmeasure,
            "rougeL": scores["rougeL"].fmeasure,
        }

    def _exact_match(self, reference: str, prediction: str) -> float:
        r = re.sub(r'\s+', ' ', reference.lower().strip())
        p = re.sub(r'\s+', ' ', prediction.lower().strip())
        return 1.0 if r == p else 0.0

    def _faithfulness(self, context: str, answer: str) -> float:
        pairs = [(context[:1200], answer[:400])]
        scores = self.nli.predict(pairs)
        probs = torch.softmax(torch.tensor(scores[0]), dim=0).numpy()
        return float(probs[2])

    def _answer_relevance(self, question: str, answer: str) -> float:
        q_emb = np.array(self.embeddings.embed_query(question))
        a_emb = np.array(self.embeddings.embed_query(answer))
        cos = np.dot(q_emb, a_emb) / (np.linalg.norm(q_emb) * np.linalg.norm(a_emb) + 1e-8)
        return float(np.clip(cos, 0, 1))

    def _context_recall(self, reference: str, retrieved_context: str) -> float:
        ref_tokens = set(nltk.word_tokenize(reference.lower()))
        ctx_tokens = set(nltk.word_tokenize(retrieved_context.lower()))
        if not ref_tokens: return 0.0
        overlap = ref_tokens & ctx_tokens
        return len(overlap) / len(ref_tokens)

    def evaluate_single(self, question: str, reference_answer: str, predicted_answer: str, context: str, guardrail_passed: bool) -> EvalResult:
        rouge = self._rouge_scores(reference_answer, predicted_answer)
        return EvalResult(
            question=question, reference_answer=reference_answer, predicted_answer=predicted_answer,
            rouge1=rouge["rouge1"], rouge2=rouge["rouge2"], rougeL=rouge["rougeL"],
            exact_match=self._exact_match(reference_answer, predicted_answer),
            faithfulness_score=self._faithfulness(context, predicted_answer),
            answer_relevance=self._answer_relevance(question, predicted_answer),
            context_recall=self._context_recall(reference_answer, context),
            passed_guardrail=guardrail_passed,
        )

    def evaluate_batch(self, qa_pairs: List[Dict], tutor_system) -> Dict[str, Any]:
        results = []
        for i, pair in enumerate(qa_pairs):
            print(f"\n  [{i+1}/{len(qa_pairs)}] Evaluating: {pair['question'][:60]}...")
            result = tutor_system.run(pair["question"], return_result=True, verbose=False)
            if result:
                eval_r = self.evaluate_single(
                    question=pair["question"], reference_answer=pair["answer"],
                    predicted_answer=result.final_answer, context=result.context_used,
                    guardrail_passed=result.guardrail_passed,
                )
                results.append(eval_r)

        if not results: return {}

        mean = lambda lst: sum(lst) / len(lst) if lst else 0.0
        metrics = {
            "num_samples": len(results),
            "rouge1": mean([r.rouge1 for r in results]),
            "rouge2": mean([r.rouge2 for r in results]),
            "rougeL": mean([r.rougeL for r in results]),
            "exact_match": mean([r.exact_match for r in results]),
            "faithfulness": mean([r.faithfulness_score for r in results]),
            "answer_relevance": mean([r.answer_relevance for r in results]),
            "context_recall": mean([r.context_recall for r in results]),
            "guardrail_pass_rate": mean([float(r.passed_guardrail) for r in results]),
            "per_sample": results,
        }
        metrics["composite_score"] = (0.25 * metrics["faithfulness"] + 0.25 * metrics["rougeL"] +
                                      0.25 * metrics["answer_relevance"] + 0.25 * metrics["context_recall"])
        return metrics

    def print_report(self, metrics: Dict[str, Any]):
        sep = "─" * 60
        print(f"\n{sep}\n  📊 EVALUATION REPORT\n{sep}")
        print(f"  Samples           : {metrics.get('num_samples', 0)}")
        print(f"  ROUGE-1           : {metrics.get('rouge1', 0):.3f}")
        print(f"  ROUGE-L           : {metrics.get('rougeL', 0):.3f}")
        print(f"  Faithfulness (NLI): {metrics.get('faithfulness', 0):.3f}")
        print(f"  Answer Relevance  : {metrics.get('answer_relevance', 0):.3f}")
        print(f"  Context Recall    : {metrics.get('context_recall', 0):.3f}")
        print(f"  Guardrail Pass %  : {metrics.get('guardrail_pass_rate', 0)*100:.1f}%")
        print(f"  ★ Composite Score : {metrics.get('composite_score', 0):.3f}\n{sep}")

    def save_report(self, metrics: Dict[str, Any], path: str = "eval_report.json"):
        export = {k: v for k, v in metrics.items() if k != "per_sample"}
        if "per_sample" in metrics:
            export["per_sample"] = [
                {"question": r.question, "reference": r.reference_answer, "predicted": r.predicted_answer,
                 "rouge1": r.rouge1, "rougeL": r.rougeL, "faithfulness": r.faithfulness_score,
                 "answer_relevance": r.answer_relevance, "guardrail_passed": r.passed_guardrail}
                for r in metrics["per_sample"]
            ]
        with open(path, "w") as f:
            json.dump(export, f, indent=2)
        logger.info(f"   💾 Eval report saved to {path}")


# ══════════════════════════════════════════════════════════════════════════════
# MODULE 6: CONVERSATION MANAGER (Multi-turn memory)
# ══════════════════════════════════════════════════════════════════════════════

class ConversationManager:
    def __init__(self, max_turns: int = 10):
        self.history: List[Dict] = []
        self.query_log: List[QueryResult] = []
        self.max_turns = max_turns

    def add_turn(self, question: str, answer: str):
        self.history.extend([{"role": "user", "content": question}, {"role": "assistant", "content": answer}])
        if len(self.history) > self.max_turns * 2:
            self.history = self.history[-(self.max_turns * 2):]

    def log_result(self, result: QueryResult):
        self.query_log.append(result)

    def save_conversation(self, path: str = "conversation_log.json"):
        data = [{"timestamp": r.timestamp, "query": r.query, "answer": r.final_answer,
                 "guardrail_passed": r.guardrail_passed, "nli_score": r.nli_score, "latency_ms": r.latency_ms}
                for r in self.query_log]
        with open(path, "w") as f:
            json.dump(data, f, indent=2)
        logger.info(f"   💾 Conversation log saved to {path}")


# ══════════════════════════════════════════════════════════════════════════════
# MODULE 7: MAIN TUTOR SYSTEM (Orchestrator)
# ══════════════════════════════════════════════════════════════════════════════

class TutorSystem:
    def __init__(self, config: Optional[TutorConfig] = None):
        self.config = config or TutorConfig()
        print(f"\n{'═'*60}")
        print("  🎓 Initializing Production AI Tutor System")
        print(f"  Device: {self.config.device}")
        print(f"{'═'*60}")

        self.loader = DocumentLoader()
        self.guardrails = GuardrailSystem(self.config)
        self.kb = KnowledgeBase(self.config)
        self.llm = LLMEngine(self.config)
        self.conversation = ConversationManager()
        self.evaluator = EvaluationSuite(self.config, self.kb.embeddings)

        print(f"\n{'═'*60}")
        print("  ✅ All modules initialized successfully!")
        print(f"{'═'*60}\n")

    def ingest_files(self, file_paths: List[str]) -> int:
        docs = self.loader.load_multiple(file_paths)
        return self.kb.ingest(docs, persist=True)

    def run(self, query: str, verbose: bool = True, return_result: bool = False) -> Optional[QueryResult]:
        if self.kb.retriever is None:
            print("❌ Knowledge base is empty. Please ingest documents first.")
            return None

        start_time = time.time()

        if verbose:
            print(f"\n{'─'*60}\n❓ STUDENT: {query}\n{'─'*60}")

        # ── Step 1: PII Sanitization ──────────────────────────────────────
        safe_query, pii_detected, pii_entities = self.guardrails.sanitize_pii(query)
        if pii_detected and verbose:
            entities = [e.entity_type for e in pii_entities]
            print(f"   ⚠️  PII Detected & Anonymized: {entities}")

        # ── Step 2: Input Guardrails (Injection & Toxicity) ───────────────
        input_safe, input_reason = self.guardrails.check_input_safety(safe_query)
        if not input_safe:
            if verbose: print(f"   🛑 INPUT BLOCKED: {input_reason}")
            blocked_answer = "I'm sorry, but your query violates safety protocols and cannot be processed."
            result = QueryResult(
                query=query, sanitized_query=safe_query, pii_detected=pii_detected, retrieved_docs=[],
                context_used="", raw_answer=blocked_answer, final_answer=blocked_answer, guardrail_passed=False,
                guardrail_reason=input_reason, nli_score=0.0, retrieval_scores=[], latency_ms=(time.time()-start_time)*1000
            )
            self.conversation.log_result(result)
            return result if return_result else None

        # ── Step 3: Hybrid Retrieval ──────────────────────────────────────
        retrieved_docs, retrieval_scores = self.kb.retriever.invoke(safe_query)

        if not retrieved_docs:
            if verbose: print("   ⚠️  No relevant documents found in knowledge base.")
            no_kb_answer = "The provided material does not contain information relevant to your question."
            return QueryResult(
                query=query, sanitized_query=safe_query, pii_detected=pii_detected, retrieved_docs=[],
                context_used="", raw_answer=no_kb_answer, final_answer=no_kb_answer, guardrail_passed=False,
                guardrail_reason="no_context", nli_score=0.0, retrieval_scores=[], latency_ms=(time.time()-start_time)*1000
            )

        if verbose:
            print(f"   📎 Retrieved {len(retrieved_docs)} chunks (reranked)")
            for i, (doc, score) in enumerate(zip(retrieved_docs, retrieval_scores)):
                src, pg = doc.metadata.get("source", "unknown"), doc.metadata.get("page", "?")
                print(f"      [{i+1}] score={score:.3f} | {src} p.{pg} | {doc.page_content[:80].replace(chr(10), ' ')}...")

        # ── Step 4: Build Context ─────────────────────────────────────────
        context_parts = []
        for i, doc in enumerate(retrieved_docs):
            src, pg = doc.metadata.get("source", "doc"), doc.metadata.get("page", "?")
            context_parts.append(f"[Source {i+1}: {Path(src).name}, page {pg}]\n{doc.page_content}")
        context = "\n\n".join(context_parts)

        # ── Step 5: Generate Answer ───────────────────────────────────────
        raw_answer = self.llm.generate(context=context, query=safe_query, history=self.conversation.history)
        if verbose: print(f"\n   🤖 Raw Answer: {raw_answer}")

        # ── Step 6: Validate Output Guardrails (Toxicity & Hallucination) ──
        passed, reason, nli_score = self.guardrails.validate_answer(context, raw_answer)

        if passed:
            final_answer = raw_answer
            if verbose: print(f"   ✅ Guardrails passed (NLI score: {nli_score:.3f})")
        else:
            final_answer = "I cannot confidently or safely answer this based on the provided material."
            if verbose: print(f"   ❌ Guardrail BLOCKED: {reason} (NLI: {nli_score:.3f})")

        # ── Step 7: Output & Logging ──────────────────────────────────────
        latency = (time.time() - start_time) * 1000
        if verbose:
            print(f"\n{'─'*60}\n🎓 TUTOR ANSWER:\n{final_answer}\n{'─'*60}\n   ⏱️  Latency: {latency:.0f}ms")

        self.conversation.add_turn(safe_query, final_answer)
        result = QueryResult(
            query=query, sanitized_query=safe_query, pii_detected=pii_detected, retrieved_docs=retrieved_docs,
            context_used=context, raw_answer=raw_answer, final_answer=final_answer, guardrail_passed=passed,
            guardrail_reason=reason, nli_score=nli_score, retrieval_scores=retrieval_scores, latency_ms=latency,
        )
        self.conversation.log_result(result)

        return result if return_result else None

    def interactive(self):
        print(f"\n{'═'*60}\n  🎓 AI Tutor — Interactive Mode\n  Commands: 'quit', 'history', 'clear'\n{'═'*60}\n")
        while True:
            try:
                query = input("You: ").strip()
            except (EOFError, KeyboardInterrupt):
                print("\nGoodbye!")
                break
            if not query: continue
            if query.lower() in ["quit", "exit", "bye"]:
                self.conversation.save_conversation()
                print("Session saved. Goodbye!")
                break
            elif query.lower() == "history":
                for i, turn in enumerate(self.conversation.history):
                    print(f"  [{i}] {turn['role'].capitalize()}: {turn['content'][:80]}...")
            elif query.lower() == "clear":
                self.conversation.history.clear()
                print("Conversation history cleared.")
            else:
                self.run(query)


def run_with_pdf(pdf_path: str):
    """Run the tutor directly with the required PDF file."""
    config = TutorConfig()
    bot = TutorSystem(config=config)

    print(f"\n📄 Loading PDF: {pdf_path}")
    if not os.path.exists(pdf_path):
        print(f"⚠️ Error: PDF not found at {pdf_path}. Please check the file path.")
        return None

    num_chunks = bot.ingest_files([pdf_path])
    print(f"✅ Loaded {num_chunks} chunks from PDF.")

    # Drop into interactive mode immediately after load
    bot.interactive()
    return bot

# ── Entry Point ───────────────────────────────────────────────────────────────
if __name__ == "__main__":
    pdf_file = "/content/2024.wat-1.5.pdf"

    print("\n" + "="*60)
    print("  🚀 STARTING PRODUCTION RAG AI TUTOR")
    print("="*60)

    run_with_pdf(pdf_file)


  🚀 STARTING PRODUCTION RAG AI TUTOR

════════════════════════════════════════════════════════════
  🎓 Initializing Production AI Tutor System
  Device: cuda
════════════════════════════════════════════════════════════


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]


════════════════════════════════════════════════════════════
  ✅ All modules initialized successfully!
════════════════════════════════════════════════════════════


📄 Loading PDF: /content/2024.wat-1.5.pdf
✅ Loaded 82 chunks from PDF.

════════════════════════════════════════════════════════════
  🎓 AI Tutor — Interactive Mode
  Commands: 'quit', 'history', 'clear'
════════════════════════════════════════════════════════════

You: what is pali dataset

────────────────────────────────────────────────────────────
❓ STUDENT: what is pali dataset
────────────────────────────────────────────────────────────
   📎 Retrieved 3 chunks (reranked)
      [1] score=6.569 | /content/2024.wat-1.5.pdf p.2 | . These datasets were selected to ensure comprehensive cover- age and accuracy i...
      [2] score=4.432 | /content/2024.wat-1.5.pdf p.5 | . In this way looking straight on and looking away from the front is seen in the...
      [3] score=1.304 | /content/2024.wat-1.5.pdf p.4 | . For Ardhamagad